In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dir = "archive/Training"
test_dir = "archive/Testing"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

print(train_generator.class_indices)

Found 5600 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.
{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [7]:
import os

print("Training folders:")
print(os.listdir("archive/Training"))

print("\nTesting folders:")
print(os.listdir("archive/Testing"))

Training folders:
['glioma', 'meningioma', 'notumor', 'pituitary']

Testing folders:
['glioma', 'meningioma', 'notumor', 'pituitary']


In [5]:
import os

for folder in ["glioma", "meningioma", "notumor", "pituitary"]:
    path = os.path.join("archive", "Training", folder)
    print(folder)
    print("Exists:", os.path.exists(path))
    if os.path.exists(path):
        print("Images:", len(os.listdir(path)))
    print("----------------")

glioma
Exists: True
Images: 1400
----------------
meningioma
Exists: True
Images: 1400
----------------
notumor
Exists: True
Images: 1400
----------------
pituitary
Exists: True
Images: 1400
----------------


In [8]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = VGG19(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers:
    layer.trainable = False

x = Flatten()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

predictions = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()



80134624/80134624 [==============================] - 31s 0us/step
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                           

In [9]:
history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=10,
    verbose=1
)

Epoch 1/10


175/175 [==============================] - 969s 6s/step - loss: 0.7962 - accuracy: 0.6630 - val_loss: 0.7187 - val_accuracy: 0.7400
Epoch 2/10
175/175 [==============================] - 1017s 6s/step - loss: 0.5619 - accuracy: 0.7723 - val_loss: 0.6357 - val_accuracy: 0.7937
Epoch 3/10
175/175 [==============================] - 786s 4s/step - loss: 0.4861 - accuracy: 0.8125 - val_loss: 0.6344 - val_accuracy: 0.7788
Epoch 4/10
175/175 [==============================] - 890s 5s/step - loss: 0.4459 - accuracy: 0.8343 - val_loss: 0.5108 - val_accuracy: 0.8306
Epoch 5/10
175/175 [==============================] - 1021s 6s/step - loss: 0.4119 - accuracy: 0.8500 - val_loss: 0.5442 - val_accuracy: 0.7975
Epoch 6/10
175/175 [==============================] - 546s 3s/step - loss: 0.3836 - accuracy: 0.8564 - val_loss: 0.5493 - val_accuracy: 0.8325
Epoch 7/10
175/175 [==============================] - 575s 3s/step - loss: 0.3748 - accuracy: 0.8570 - val_loss: 0.5371 - val_accuracy: 0.

In [10]:
model.save("final_model.keras")
print("Model saved successfully!")

Model saved successfully!
